In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvcc --version


In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126


In [ ]:
!pip install uv

In [ ]:
!uv pip install surya-ocr


In [ ]:
!uv pip install pypdfium2

# extract folder of files

In [ ]:
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
import pypdfium2 as pdfium
import os


In [ ]:
# Initialize predictors once (outside the loop for efficiency)
foundation_predictor = FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()

In [ ]:
input_folder = "/content/drive/MyDrive/Graduation Project/Books"
output_folder = "/content/drive/MyDrive/Graduation Project/Collected Data"

pdf_files = [f for f in os.listdir(input_folder) if f.endswith('.pdf')]
print(f"Found {len(pdf_files)} PDF files to process\n")


In [ ]:
# Process each PDF file
for file_index, pdf_filename in enumerate(pdf_files, 1):
    pdf_path = os.path.join(input_folder, pdf_filename)

    # Create output filename (replace .pdf with .txt)
    output_filename = pdf_filename.replace('.pdf', '.txt')
    output_path = os.path.join(output_folder, output_filename)

    print(f"\n{'='*60}")
    print(f"[{file_index}/{len(pdf_files)}] Processing: {pdf_filename}")
    print(f"{'='*60}")

    try:
        # Open PDF
        pdf = pdfium.PdfDocument(pdf_path)
        n_pages = len(pdf)
        print(f"Total Pages: {n_pages}")
        print("Starting text extraction...\n")

        # Store all extracted text
        all_text = []

        # Loop through all pages (or first 35 if you want)
        for page_num in range(n_pages):
            try:
                print(f"  Processing page {page_num + 1}/{n_pages}...", end=" ")

                # Render page
                page = pdf[page_num]
                bitmap = page.render(scale=1, rotation=0)
                pil_image = bitmap.to_pil()

                # Step 1: Detect all text boxes on the page
                detections = detection_predictor([pil_image])
                detected_boxes = detections[0].bboxes

                # Step 2: Recognize text in each detected box
                if len(detected_boxes) > 0:
                    predictions = recognition_predictor([pil_image], det_predictor=detection_predictor)
                    page_text_lines = [predictions[0].text_lines[i].text for i in range(len(predictions[0].text_lines))]
                else:
                    page_text_lines = []

                # Store text from this page
                all_text.append({
                    'page': page_num + 1,
                    'full_text': '\n'.join(page_text_lines)
                })

                print(f"({len(detected_boxes)} boxes, {len(page_text_lines)} lines)")

            except Exception as e:
                print(f"Error: {str(e)}")
                continue

        # Save to file
        with open(output_path, 'w', encoding='utf-8') as f:
            for page_data in all_text:
                f.write(f"\n{'='*50}\n")
                f.write(f"PAGE {page_data['page']}\n")
                f.write(f"{'='*50}\n")
                f.write(page_data['full_text'])
                f.write("\n")

        print(f"\n Extraction complete! Processed {len(all_text)} pages")
        print(f"Text saved to: {output_path}")

    except Exception as e:
        print(f"\n✗ Error processing file {pdf_filename}: {str(e)}")
        continue

print(f"\n{'='*60}")
print("✓ All files processed successfully!")
print(f"{'='*60}")

# extract single file

In [ ]:
# pdf_path ="/content/كل_ما_يخص_ايصالات_الامانة.pdf"

# pdf = pdfium.PdfDocument(pdf_path)
# n_pages = len(pdf)

# print(f"Total Pages: {n_pages}")
# print("Starting text extraction...\n")



# # Store all extracted text
# all_text = []

# # Loop through first 35 pages only
# for page_num in range(1,n_pages):
#     try:
#         print(f"Processing page {page_num + 1}/{n_pages}...")

#         # Render page
#         page = pdf[page_num]
#         bitmap = page.render(
#             scale=1,
#             rotation=0,
#         )
#         pil_image = bitmap.to_pil()

#         # Step 1: Detect all text boxes on the page
#         detections = detection_predictor([pil_image])
#         detected_boxes = detections[0].bboxes  # Get all bounding boxes

#         print(f"  Found {len(detected_boxes)} text boxes")

#         # Step 2: Recognize text in each detected box
#         if len(detected_boxes) > 0:
#             predictions = recognition_predictor([pil_image], det_predictor=detection_predictor)
#             page_text_lines = [predictions[0].text_lines[i].text for i in range(len(predictions[0].text_lines))]
#         else:
#             page_text_lines = []

#         # Store text from this page
#         all_text.append({
#             'page': page_num + 1,
#             'full_text': '\n'.join(page_text_lines)
#         })

#         print(f"  ✓ Extracted {len(page_text_lines)} text lines from {len(detected_boxes)} boxes")

#     except Exception as e:
#         print(f"  ✗ Error processing page {page_num + 1}: {str(e)}")
#         continue

# print(f"\n{'='*50}")
# print(f"Extraction complete! Processed {len(all_text)} pages")
# print(f"{'='*50}\n")

# # Print results
# for page_data in all_text:
#     print(f"\n--- PAGE {page_data['page']}")
#     print(page_data['full_text'])
#     print("-" * 50)

# # Optional: Save to file
# output_file = "/content/text2.txt"
# with open(output_file, 'w', encoding='utf-8') as f:
#     for page_data in all_text:
#         f.write(f"\n{'='*50}\n")
#         f.write(f"PAGE {page_data['page']}\n")
#         f.write(f"{'='*50}\n")
#         f.write(page_data['full_text'])
#         f.write("\n")

# print(f"\nText saved to '{output_file}'")